# 📊 Data Preparation - Assistente Virtual Médico Generalista

**Projeto:** Tech Challenge Fase 3 - FIAP/Alura  
**Objetivo:** Preparar dados médicos para fine-tuning de LLM com conformidade LGPD  
**Data:** Março de 2026

---

## 📋 Sumário

1. [Setup e Configuração](#1-setup-e-configuração)
2. [Carregamento de Dados](#2-carregamento-de-dados)
3. [Análise Exploratória](#3-análise-exploratória)
4. [Anonimização LGPD](#4-anonimização-lgpd)
5. [Sanitização de Conteúdo](#5-sanitização-de-conteúdo)
6. [Formatação para Hugging Face](#6-formatação-para-hugging-face)
7. [Validação de Qualidade](#7-validação-de-qualidade)
8. [Visualizações](#8-visualizações)
9. [Exportação Final](#9-exportação-final)
10. [Conclusão e Próximos Passos](#10-conclusão-e-próximos-passos)

## 1. Setup e Configuração

### 1.1 Importação de Bibliotecas

In [ ]:
# Bibliotecas padrão
import os
import sys
import json
import re
import random
from pathlib import Path
from datetime import datetime
from collections import Counter
from typing import List, Dict, Any, Tuple, Optional
import warnings

# Data manipulation
import pandas as pd
import numpy as np

# Hugging Face
from datasets import Dataset, DatasetDict, load_dataset

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Progresso
from tqdm.notebook import tqdm

# Configurações
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

%matplotlib inline

print("✅ Bibliotecas importadas com sucesso!")

In [ ]:
# Tentar importar bibliotecas opcionais
try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
    print("✅ WordCloud disponível")
except ImportError:
    WORDCLOUD_AVAILABLE = False
    print("⚠️ WordCloud não instalado - algumas visualizações serão ignoradas")

try:
    from dotenv import load_dotenv
    load_dotenv()
    print("✅ Variáveis de ambiente carregadas")
except ImportError:
    print("⚠️ python-dotenv não instalado - usando configurações padrão")

### 1.2 Configuração de Caminhos e Parâmetros

In [ ]:
# Caminhos do projeto
PROJECT_ROOT = Path("..")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Criar diretórios se não existirem
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Parâmetros de processamento
CONFIG = {
    "max_seq_length": 512,
    "min_output_length": 10,
    "max_output_length": 2048,
    "train_split": 0.8,
    "val_split": 0.1,
    "test_split": 0.1,
    "random_seed": 42,
    "output_file": "medical_data_unified.jsonl"
}

# Arquivos de saída
OUTPUT_PATH = PROCESSED_DATA_DIR / CONFIG["output_file"]
METRICS_PATH = PROCESSED_DATA_DIR / "preparation_metrics.json"
SANITIZATION_REPORT_PATH = PROCESSED_DATA_DIR / "sanitization_report.json"

# Seed para reprodutibilidade
random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])

print(f"📁 Diretório de dados brutos: {RAW_DATA_DIR.absolute()}")
print(f"📁 Diretório de dados processados: {PROCESSED_DATA_DIR.absolute()}")
print(f"⚙️ Configurações: {CONFIG}")

---

## 2. Carregamento de Dados

### 2.1 Verificar Arquivos Existentes

In [ ]:
def list_jsonl_files(directory: Path) -> List[Path]:
    """Lista todos os arquivos JSONL em um diretório."""
    jsonl_files = list(directory.glob("*.jsonl"))
    return jsonl_files

existing_files = list_jsonl_files(RAW_DATA_DIR)
print(f"📂 Arquivos JSONL encontrados em {RAW_DATA_DIR}: {len(existing_files)}")

for f in existing_files:
    size_kb = f.stat().st_size / 1024
    print(f"  - {f.name} ({size_kb:.2f} KB)")

### 2.2 Criar Dados de Exemplo (Fallback)

Se não houver dados brutos, criamos um conjunto de exemplo para demonstração do pipeline.

In [ ]:
# Dados de exemplo baseados nas 3 fontes do projeto
SAMPLE_DATA = {
    "protocolos_medicos.jsonl": [
        {
            "instruction": "Qual o protocolo de tratamento para hipertensão arterial?",
            "input": "",
            "output": "O tratamento da hipertensão arterial segue as diretrizes do Ministério da Saúde. Paciente João Silva, CPF 123.456.789-00, diagnosticado em 15/03/2024 no Hospital das Clínicas. Iniciar com modificações no estilo de vida: redução do consumo de sódio, prática de atividade física regular (150 min/semana), controle do peso corporal. Se PA > 140/90 mmHg após 3 meses, iniciar terapia farmacológica com IECA ou BRA. Reavaliação em 30 dias. Dr. Carlos Mendes - CRM 12345/SP."
        },
        {
            "instruction": "Como tratar diabetes mellitus tipo 2?",
            "input": "Paciente adulto, recém-diagnosticado",
            "output": "O tratamento do diabetes mellitus tipo 2 em adultos recém-diagnosticados deve ser iniciado com orientações sobre mudanças no estilo de vida. A paciente Maria Oliveira, nascida em 20/05/1975, atendida na UBS Centro pelo Dr. Pedro Santos (CRM 54321/RJ), deve iniciar metformina 500mg 2x/dia, com aumento gradual. Meta de HbA1c < 7%. Monitorar glicemia capilar em jejum. Contato: (21) 99999-8888. Endereço: Rua das Flores, 123, Centro."
        },
        {
            "instruction": "Protocolo de atendimento para dor torácica aguda",
            "input": "",
            "output": "Dor torácica aguda requer avaliação imediata. TRIAGEM: ECG em até 10 minutos. Se elevação de ST, acionar protocolo de IAM. Paciente atendido: Roberto Costa, RG 12.345.678-9, 58 anos, encaminhado ao Hospital São Lucas. Administrar AAS 200mg VO, O2 se SatO2 < 94%, acesso venoso periférico. Monitorização contínua. Coletar troponina e enzimas cardíacas. Avaliação pelo cardiologista Dr. Ana Paula - telefone (11) 3333-4444."
        },
        {
            "instruction": "Manejo de infecção urinária não complicada",
            "input": "Mulher jovem, sem comorbidades",
            "output": "ITU não complicada em mulheres: Nitrofurantoína 100mg 6/6h por 5-7 dias OU Fosfomicina 3g dose única. Orientar hidratação adequada (2-3L/dia). Paciente Juliana Ferreira, CNS 123456789012345, consulta realizada em 10/02/2025 na Clínica Santa Maria. Se sintomas persistirem > 48h, reavaliar e considerar urocultura. Evitar atividade sexual durante tratamento. Retorno se febre > 38°C ou dor lombar."
        },
        {
            "instruction": "Tratamento de asma leve persistente",
            "input": "",
            "output": "Asma leve persistente: Corticoide inalatório em dose baixa (Budesonida 200-400mcg/dia ou Beclometasona 200-500mcg/dia). Beta-agonista de curta ação para resgate. Paciente Marcos Almeida, 25 anos, morador da Av. Paulista 1000, apt 501, São Paulo-SP. Plano de ação por escrito. Vacinação contra Influenza anual. Evitar gatilhos conhecidos. Espirometria de controle em 3 meses no Hospital Albert Einstein. Dra. Beatriz Lima - CRM 67890/SP."
        }
    ],
    "perguntas_frequentes.jsonl": [
        {
            "instruction": "Quais são os sintomas da gripe?",
            "input": "",
            "output": "Os principais sintomas da gripe incluem: febre alta (acima de 38°C), dor de cabeça intensa, dores musculares e articulares, fadiga extrema, tosse seca, dor de garganta e coriza. Em casos mais graves, pode haver dificuldade respiratória. Diferentemente do resfriado comum, a gripe tem início súbito e sintomas mais intensos. Procure atendimento médico se houver febre persistente por mais de 3 dias ou dificuldade para respirar."
        },
        {
            "instruction": "Como prevenir a dengue?",
            "input": "",
            "output": "A prevenção da dengue envolve eliminar criadouros do mosquito Aedes aegypti: não deixar água parada em pneus, vasos de plantas, garrafas e recipientes; manter caixas d'água tampadas; usar repelente e roupas que cubram braços e pernas; instalar telas em janelas. O morador José Pereira da Rua do Sol, 456, bairro Boa Vista, foi notificado pela vigilância sanitária em 25/01/2025. Procure a UBS mais próxima se apresentar febre, dor atrás dos olhos e manchas vermelhas."
        },
        {
            "instruction": "Quando devo tomar vacina contra COVID-19?",
            "input": "",
            "output": "O esquema vacinal contra COVID-19 segue as recomendações do PNI. Dose de reforço recomendada anualmente, especialmente para grupos de risco: idosos > 60 anos, imunossuprimidos, gestantes e profissionais de saúde. Compareça à unidade de saúde portando documento de identidade e cartão de vacinação. Informações adicionais pelo telefone 136 ou no site do Ministério da Saúde. Intervalo mínimo de 4 meses da última dose."
        },
        {
            "instruction": "Quais exames fazer no pré-natal?",
            "input": "Primeira gestação",
            "output": "No pré-natal da primeira gestação, são solicitados: hemograma completo, tipagem sanguínea e fator Rh, glicemia de jejum, sorologias (HIV, sífilis, hepatite B, toxoplasmose, rubéola), exame de urina, ultrassonografia obstétrica. A gestante Amanda Santos, nascida em 12/08/1995, CPF 987.654.321-00, deve realizar consultas mensais até 28 semanas, quinzenais até 36 semanas e semanais até o parto. Acompanhamento na UBS Jardim das Flores com Dra. Carla Dias."
        },
        {
            "instruction": "Como identificar sinais de AVC?",
            "input": "",
            "output": "Use a regra SAMU: Sorriso (assimétrico?), Abraço (um braço não levanta?), Mensagem (fala enrolada?) e Urgente (ligue 192). Outros sinais incluem: confusão mental súbita, dificuldade para enxergar, dor de cabeça intensa sem causa aparente, tontura e perda de equilíbrio. O AVC é uma emergência médica - cada minuto sem tratamento causa morte de milhões de neurônios. Tempo é cérebro. Procure atendimento imediato."
        }
    ],
    "modelos_laudos.jsonl": [
        {
            "instruction": "Gere um modelo de laudo de radiografia de tórax normal",
            "input": "",
            "output": "LAUDO RADIOLÓGICO - RADIOGRAFIA DE TÓRAX\n\nPaciente: Fernando Gomes de Souza\nData de nascimento: 03/11/1980\nData do exame: 28/02/2025\nSolicitante: Dr. Ricardo Teixeira - CRM 11111/MG\n\nTÉCNICA: Radiografia de tórax em PA e perfil.\n\nACHADOS:\n- Campos pulmonares com transparência preservada\n- Ausência de consolidações ou infiltrados\n- Seios costofrênicos livres bilateralmente\n- Índice cardiotorácico dentro dos limites da normalidade\n- Mediastino centrado, sem alargamento\n- Estruturas ósseas sem alterações\n\nIMPRESSÃO: Radiografia de tórax sem alterações significativas.\n\nRadiologista: Dra. Paula Mendonça - CRM 22222/MG"
        },
        {
            "instruction": "Modelo de laudo de ultrassonografia abdominal",
            "input": "Investigação de dor abdominal",
            "output": "LAUDO DE ULTRASSONOGRAFIA ABDOMINAL TOTAL\n\nPaciente: Antônio Carlos Ribeiro\nIdade: 45 anos\nProntuário: 123456\nData: 01/03/2025\n\nINDICAÇÃO CLÍNICA: Dor abdominal a esclarecer\n\nACHADOS:\nFÍGADO: Dimensões normais, ecotextura homogênea, contornos regulares. Ausência de lesões focais.\nVESÍCULA BILIAR: Paredes finas, sem cálculos.\nVIAS BILIARES: Calibre normal (colédoco: 4mm).\nPÂNCREAS: Visualização parcial, sem alterações evidentes.\nBAÇO: Dimensões e ecotextura normais.\nRINS: Simétricos, contornos regulares, diferenciação corticomedular preservada. Ausência de dilatação pielocalicial ou litíase.\nAORTA: Calibre normal.\n\nIMPRESSÃO: Exame ultrassonográfico sem alterações significativas.\n\nDr. Marcos Vinícius Alves - CRM 33333/BA"
        },
        {
            "instruction": "Laudo de tomografia computadorizada de crânio",
            "input": "Cefaleia intensa",
            "output": "LAUDO DE TOMOGRAFIA COMPUTADORIZADA DE CRÂNIO\n\nPaciente: Luciana Ferreira da Silva\nIdade: 38 anos\nRG: 98.765.432-1\nData do exame: 05/03/2025\nEquipamento: TC multidetectores 64 canais\n\nTÉCNICA: TC de crânio sem contraste, cortes axiais de 5mm.\n\nACHADOS:\n- Parênquima encefálico com densidade e morfologia preservadas\n- Sistema ventricular de dimensões normais\n- Estruturas da linha média centradas\n- Ausência de coleções extra-axiais\n- Cisternas basais livres\n- Estruturas ósseas íntegras\n- Seios paranasais e mastoideias aerados\n\nIMPRESSÃO: Tomografia de crânio dentro dos padrões de normalidade.\n\nOBSERVAÇÃO: Correlacionar com dados clínicos. Se necessário, complementar com RM para avaliação mais detalhada.\n\nDra. Cristina Moraes - CRM 44444/SP"
        },
        {
            "instruction": "Modelo de laudo de ressonância magnética de coluna lombar",
            "input": "",
            "output": "LAUDO DE RESSONÂNCIA MAGNÉTICA DA COLUNA LOMBAR\n\nPaciente: Eduardo Martins Pereira\nData de nascimento: 15/07/1965\nConvênio: Unimed - Carteira 9876543210\nData: 02/03/2025\n\nTÉCNICA: Sequências ponderadas em T1, T2 e STIR nos planos sagital e axial.\n\nACHADOS:\n- Alinhamento vertebral preservado\n- Corpos vertebrais com altura e sinal normais\n- L4-L5: Protrusão discal posterior central, com contato da face ventral do saco dural, sem compressão radicular significativa\n- L5-S1: Desidratação discal (hipossinal em T2), sem herniações\n- Canal vertebral de amplitude preservada\n- Cone medular com sinal e morfologia normais, terminando em L1\n- Facetas articulares com sinais de artropatia inicial\n\nIMPRESSÃO:\n1. Protrusão discal L4-L5\n2. Discopatia degenerativa L5-S1\n\nDr. Roberto Campos - CRM 55555/PR\nEndereço: Clínica Diagnósticos Avançados, Rua Principal 789, Curitiba-PR\nTelefone: (41) 3456-7890"
        },
        {
            "instruction": "Laudo de eletrocardiograma",
            "input": "Check-up cardiológico",
            "output": "LAUDO DE ELETROCARDIOGRAMA\n\nPaciente: Regina Helena Costa\nIdade: 52 anos\nCPF: 456.789.123-00\nData do exame: 04/03/2025\nSolicitante: Dra. Fernanda Lopes - CRM 66666/RJ\n\nDADOS TÉCNICOS:\n- Velocidade: 25mm/s\n- Amplitude: 10mm/mV\n- 12 derivações padrão\n\nANÁLISE:\n- Ritmo: Sinusal\n- Frequência cardíaca: 72 bpm\n- Eixo elétrico: Normal (entre 0° e +90°)\n- Onda P: Duração e amplitude normais\n- Intervalo PR: 160ms (normal)\n- Complexo QRS: Duração 88ms, morfologia normal\n- Segmento ST: Isoelétrico\n- Onda T: Positiva nas derivações esperadas\n- Intervalo QT corrigido: 420ms (normal)\n\nCONCLUSÃO: Eletrocardiograma dentro dos limites da normalidade.\n\nCardiologista: Dr. Alexandre Nunes - CRM 77777/RJ\nEndereço: Av. Atlântica 500, sala 301, Copacabana, Rio de Janeiro\nEmail: cardioclinica@email.com"
        }
    ]
}

print(f"📋 Dados de exemplo preparados:")
for source, records in SAMPLE_DATA.items():
    print(f"  - {source}: {len(records)} registros")

In [ ]:
def create_sample_data_files(data: Dict[str, List[Dict]], directory: Path) -> List[Path]:
    """Cria arquivos JSONL de exemplo se não existirem dados brutos."""
    created_files = []
    
    for filename, records in data.items():
        filepath = directory / filename
        
        if not filepath.exists():
            with open(filepath, 'w', encoding='utf-8') as f:
                for record in records:
                    f.write(json.dumps(record, ensure_ascii=False) + '\n')
            created_files.append(filepath)
            print(f"✅ Criado: {filepath.name}")
        else:
            print(f"ℹ️ Já existe: {filepath.name}")
    
    return created_files

# Criar arquivos de exemplo se necessário
if len(existing_files) == 0:
    print("\n⚠️ Nenhum dado bruto encontrado. Criando dados de exemplo...\n")
    create_sample_data_files(SAMPLE_DATA, RAW_DATA_DIR)
    existing_files = list_jsonl_files(RAW_DATA_DIR)

### 2.3 Carregar Dados de Múltiplas Fontes

In [ ]:
def load_jsonl_file(filepath: Path) -> List[Dict]:
    """Carrega um arquivo JSONL e retorna lista de dicionários."""
    records = []
    errors = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
                record['_source_file'] = filepath.name
                record['_line_number'] = line_num
                records.append(record)
            except json.JSONDecodeError as e:
                errors.append({'line': line_num, 'error': str(e)})
    
    if errors:
        print(f"⚠️ {len(errors)} erros de parsing em {filepath.name}")
    
    return records

def load_all_data(directory: Path) -> List[Dict]:
    """Carrega todos os arquivos JSONL de um diretório."""
    all_records = []
    
    for filepath in sorted(directory.glob("*.jsonl")):
        records = load_jsonl_file(filepath)
        all_records.extend(records)
        print(f"📄 {filepath.name}: {len(records)} registros carregados")
    
    return all_records

# Carregar todos os dados
print("\n🔄 Carregando dados brutos...\n")
raw_data = load_all_data(RAW_DATA_DIR)
print(f"\n📊 Total de registros carregados: {len(raw_data)}")

---

## 3. Análise Exploratória

### 3.1 Estatísticas Descritivas

In [ ]:
# Converter para DataFrame para análise
df = pd.DataFrame(raw_data)

print("📋 Estrutura do Dataset:")
print(f"   - Total de registros: {len(df)}")
print(f"   - Colunas: {list(df.columns)}")
print()

# Verificar campos obrigatórios
required_fields = ['instruction', 'input', 'output']
for field in required_fields:
    if field in df.columns:
        non_null = df[field].notna().sum()
        non_empty = (df[field].fillna('') != '').sum()
        print(f"   ✅ {field}: {non_null} não-nulos, {non_empty} não-vazios")
    else:
        print(f"   ❌ {field}: CAMPO AUSENTE!")

In [ ]:
# Estatísticas de texto
def get_text_stats(series: pd.Series) -> Dict:
    """Calcula estatísticas de uma série de textos."""
    lengths_chars = series.fillna('').str.len()
    lengths_words = series.fillna('').str.split().str.len()
    
    return {
        'count': len(series),
        'non_empty': (series.fillna('') != '').sum(),
        'avg_chars': lengths_chars.mean(),
        'median_chars': lengths_chars.median(),
        'max_chars': lengths_chars.max(),
        'min_chars': lengths_chars.min(),
        'avg_words': lengths_words.mean(),
        'max_words': lengths_words.max()
    }

print("\n📈 Estatísticas de Texto:\n")
for field in ['instruction', 'input', 'output']:
    if field in df.columns:
        stats = get_text_stats(df[field])
        print(f"📝 {field.upper()}:")
        print(f"   - Registros não-vazios: {stats['non_empty']}")
        print(f"   - Média de caracteres: {stats['avg_chars']:.1f}")
        print(f"   - Média de palavras: {stats['avg_words']:.1f}")
        print(f"   - Máximo de palavras: {stats['max_words']:.0f}")
        print()

In [ ]:
# Distribuição por fonte
if '_source_file' in df.columns:
    print("📊 Distribuição por Fonte:\n")
    source_counts = df['_source_file'].value_counts()
    for source, count in source_counts.items():
        pct = count / len(df) * 100
        print(f"   - {source}: {count} registros ({pct:.1f}%)")

### 3.2 Visualização de Amostras

In [ ]:
# Exibir amostras de cada fonte
print("\n📄 Amostras dos Dados (1 por fonte):\n")
print("=" * 80)

if '_source_file' in df.columns:
    for source in df['_source_file'].unique():
        sample = df[df['_source_file'] == source].iloc[0]
        print(f"\n📁 Fonte: {source}")
        print("-" * 40)
        print(f"Instruction: {sample['instruction'][:200]}..." if len(sample['instruction']) > 200 else f"Instruction: {sample['instruction']}")
        print(f"Input: {sample['input'] if sample['input'] else '(vazio)'}")
        print(f"Output: {sample['output'][:300]}..." if len(sample['output']) > 300 else f"Output: {sample['output']}")
        print("=" * 80)

---

## 4. Anonimização LGPD

### 4.1 Definição de Padrões de PII (Personally Identifiable Information)

In [ ]:
class PIIAnonymizer:
    """Classe para anonimização de dados pessoais conforme LGPD."""
    
    # Padrões regex para detecção de PII
    PATTERNS = {
        'cpf': {
            'pattern': r'\b\d{3}\.\d{3}\.\d{3}-\d{2}\b',
            'replacement': '[CPF_ANONIMIZADO]',
            'description': 'CPF no formato XXX.XXX.XXX-XX'
        },
        'cpf_digits': {
            'pattern': r'\b\d{11}\b(?!\d)',  # 11 dígitos seguidos (CPF sem formatação)
            'replacement': '[CPF_ANONIMIZADO]',
            'description': 'CPF sem formatação (11 dígitos)'
        },
        'rg': {
            'pattern': r'\b\d{2}\.\d{3}\.\d{3}-\d{1}\b',
            'replacement': '[RG_ANONIMIZADO]',
            'description': 'RG no formato XX.XXX.XXX-X'
        },
        'cns': {
            'pattern': r'\b\d{15}\b',
            'replacement': '[CNS_ANONIMIZADO]',
            'description': 'Cartão Nacional de Saúde (15 dígitos)'
        },
        'crm': {
            'pattern': r'\bCRM\s*\d{4,6}[/-]?[A-Z]{2}\b',
            'replacement': '[CRM_ANONIMIZADO]',
            'description': 'Número de registro CRM'
        },
        'telefone_com_ddd': {
            'pattern': r'\(\d{2}\)\s*\d{4,5}-?\d{4}\b',
            'replacement': '[TELEFONE_ANONIMIZADO]',
            'description': 'Telefone com DDD'
        },
        'telefone_simples': {
            'pattern': r'\b\d{4,5}-\d{4}\b',
            'replacement': '[TELEFONE_ANONIMIZADO]',
            'description': 'Telefone sem DDD'
        },
        'email': {
            'pattern': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
            'replacement': '[EMAIL_ANONIMIZADO]',
            'description': 'Endereço de email'
        },
        'data_nascimento': {
            'pattern': r'\b(nascid[oa]?\s+(em|:)?\s*)?\d{2}[/-]\d{2}[/-]\d{4}\b',
            'replacement': '[DATA_ANONIMIZADA]',
            'description': 'Data no formato DD/MM/AAAA ou DD-MM-AAAA'
        },
        'data_exame': {
            'pattern': r'\b\d{2}[/-]\d{2}[/-]\d{4}\b',
            'replacement': '[DATA_ANONIMIZADA]',
            'description': 'Data genérica'
        },
        'endereco_rua': {
            'pattern': r'\b(Rua|Av\.?|Avenida|Alameda|Travessa|Praça)\s+[A-Za-zÀ-ú\s]+,?\s*\d{1,5}(,?\s*(apt|apto|sala|bloco|casa)?\s*\d{1,5})?',
            'replacement': '[ENDERECO_ANONIMIZADO]',
            'description': 'Endereço com rua/avenida e número'
        },
        'cep': {
            'pattern': r'\b\d{5}-?\d{3}\b',
            'replacement': '[CEP_ANONIMIZADO]',
            'description': 'CEP'
        },
        'prontuario': {
            'pattern': r'\b(prontu[aá]rio|carteira|matr[ií]cula)[:\s]*\d{4,10}\b',
            'replacement': '[PRONTUARIO_ANONIMIZADO]',
            'description': 'Número de prontuário ou matrícula'
        },
        'nome_paciente': {
            'pattern': r'\b[Pp]aciente[:\s]+([A-Z][a-zà-ú]+\s+){1,4}([A-Z][a-zà-ú]+)',
            'replacement': 'Paciente [NOME_ANONIMIZADO]',
            'description': 'Nome de paciente'
        },
        'nome_dr': {
            'pattern': r'\b[Dd]r[a]?\.?\s+([A-Z][a-zà-ú]+\s+){1,3}([A-Z][a-zà-ú]+)',
            'replacement': '[MEDICO_ANONIMIZADO]',
            'description': 'Nome de médico'
        },
        'nome_completo': {
            'pattern': r'\b([A-Z][a-zà-ú]+\s+){2,4}([A-Z][a-zà-ú]+)\b(?=,|\.|\s+-|\s+CPF|\s+RG|\s+nascid)',
            'replacement': '[NOME_ANONIMIZADO]',
            'description': 'Nome completo seguido de identificador'
        },
        'hospital_clinica': {
            'pattern': r'\b(Hospital|Cl[ií]nica|UBS|UPA|Ambulat[óo]rio)\s+([A-Z][a-zà-ú]+\s*){1,4}',
            'replacement': '[INSTITUICAO_ANONIMIZADA]',
            'description': 'Nome de instituição de saúde'
        }
    }
    
    def __init__(self):
        self.stats = {key: 0 for key in self.PATTERNS.keys()}
        self.total_anonymized = 0
    
    def anonymize_text(self, text: str) -> Tuple[str, Dict[str, int]]:
        """Anonimiza texto e retorna o texto processado e contagem de substituições."""
        if not text or not isinstance(text, str):
            return text, {}
        
        anonymized = text
        local_stats = {}
        
        for pii_type, config in self.PATTERNS.items():
            pattern = re.compile(config['pattern'], re.IGNORECASE)
            matches = pattern.findall(anonymized)
            count = len(matches)
            
            if count > 0:
                anonymized = pattern.sub(config['replacement'], anonymized)
                local_stats[pii_type] = count
                self.stats[pii_type] += count
                self.total_anonymized += count
        
        return anonymized, local_stats
    
    def anonymize_record(self, record: Dict) -> Dict:
        """Anonimiza todos os campos de texto de um registro."""
        anonymized_record = record.copy()
        
        for field in ['instruction', 'input', 'output']:
            if field in record and record[field]:
                anonymized_record[field], _ = self.anonymize_text(record[field])
        
        return anonymized_record
    
    def get_report(self) -> Dict:
        """Retorna relatório de anonimização."""
        return {
            'total_entities_anonymized': self.total_anonymized,
            'by_type': {k: v for k, v in self.stats.items() if v > 0},
            'patterns_used': len([k for k, v in self.stats.items() if v > 0])
        }

# Criar instância do anonimizador
anonymizer = PIIAnonymizer()

print("✅ Anonimizador configurado com os seguintes padrões:\n")
for pii_type, config in anonymizer.PATTERNS.items():
    print(f"   - {pii_type}: {config['description']}")

### 4.2 Demonstração de Anonimização

In [ ]:
# Demonstrar anonimização com exemplo
demo_anonymizer = PIIAnonymizer()

demo_text = """
Paciente João Silva de Oliveira, CPF 123.456.789-00, nascido em 15/03/1980.
Atendido no Hospital das Clínicas pelo Dr. Carlos Mendes (CRM 12345/SP).
Endereço: Rua das Flores, 123, apt 501, São Paulo-SP.
Contato: (11) 99999-8888, email: joao.silva@email.com
Prontuário: 123456
"""

anonymized_demo, demo_stats = demo_anonymizer.anonymize_text(demo_text)

print("📝 DEMONSTRAÇÃO DE ANONIMIZAÇÃO\n")
print("=" * 60)
print("TEXTO ORIGINAL:")
print("-" * 60)
print(demo_text)
print("\n" + "=" * 60)
print("TEXTO ANONIMIZADO:")
print("-" * 60)
print(anonymized_demo)
print("\n" + "=" * 60)
print("ESTATÍSTICAS:")
print("-" * 60)
for pii_type, count in demo_stats.items():
    print(f"   - {pii_type}: {count} ocorrência(s) removida(s)")

### 4.3 Aplicar Anonimização aos Dados

In [ ]:
# Aplicar anonimização a todos os registros
print("🔒 Aplicando anonimização LGPD aos dados...\n")

anonymized_data = []
for record in tqdm(raw_data, desc="Anonimizando"):
    anonymized_record = anonymizer.anonymize_record(record)
    anonymized_data.append(anonymized_record)

# Relatório de anonimização
anon_report = anonymizer.get_report()

print("\n✅ Anonimização concluída!")
print(f"\n📊 Relatório de Anonimização:")
print(f"   - Total de entidades anonimizadas: {anon_report['total_entities_anonymized']}")
print(f"   - Tipos de PII encontrados: {anon_report['patterns_used']}")
print("\n   Detalhamento por tipo:")
for pii_type, count in sorted(anon_report['by_type'].items(), key=lambda x: -x[1]):
    print(f"      - {pii_type}: {count}")

### 4.4 Verificação de Anonimização

In [ ]:
# Comparar antes e depois (amostras)
print("\n🔍 Comparação Antes/Depois da Anonimização:\n")
print("=" * 80)

for i in range(min(3, len(raw_data))):
    print(f"\n📄 Registro {i+1}:")
    print("-" * 40)
    print("ANTES (output):")
    print(raw_data[i]['output'][:400] + "..." if len(raw_data[i]['output']) > 400 else raw_data[i]['output'])
    print("\nDEPOIS (output):")
    print(anonymized_data[i]['output'][:400] + "..." if len(anonymized_data[i]['output']) > 400 else anonymized_data[i]['output'])
    print("=" * 80)

---

## 5. Sanitização de Conteúdo

### 5.1 Funções de Limpeza e Normalização

In [ ]:
class ContentSanitizer:
    """Classe para sanitização e validação de conteúdo médico."""
    
    # Palavras/frases que indicam conteúdo potencialmente perigoso
    DANGEROUS_PATTERNS = [
        r'\b(autom[eé]dica[çc][aã]o|automedique)\b',
        r'\b(n[aã]o\s+procure\s+m[eé]dico)\b',
        r'\b(dispensa\s+acompanhamento)\b',
        r'\b(n[aã]o\s+[eé]\s+necess[aá]rio\s+consultar)\b',
    ]
    
    # Disclaimer médico padrão
    MEDICAL_DISCLAIMER = (
        "\n\n⚠️ IMPORTANTE: Esta informação é apenas para fins educacionais. "
        "Consulte sempre um profissional de saúde para diagnóstico e tratamento adequados."
    )
    
    def __init__(self, config: Dict):
        self.config = config
        self.stats = {
            'total_processed': 0,
            'normalized': 0,
            'truncated': 0,
            'removed_empty': 0,
            'flagged_dangerous': 0,
            'added_disclaimer': 0
        }
    
    def normalize_text(self, text: str) -> str:
        """Normaliza texto: espaços, quebras de linha, encoding."""
        if not text:
            return ""
        
        # Normalizar espaços múltiplos
        text = re.sub(r'\s+', ' ', text)
        
        # Remover espaços no início/fim
        text = text.strip()
        
        # Normalizar quebras de linha
        text = re.sub(r'\n\s*\n+', '\n\n', text)
        
        # Remover caracteres de controle (exceto newline e tab)
        text = ''.join(char for char in text if char.isprintable() or char in '\n\t')
        
        return text
    
    def truncate_text(self, text: str, max_words: int) -> str:
        """Trunca texto se exceder o limite de palavras."""
        words = text.split()
        if len(words) <= max_words:
            return text
        
        truncated = ' '.join(words[:max_words])
        self.stats['truncated'] += 1
        return truncated + "..."
    
    def check_dangerous_content(self, text: str) -> bool:
        """Verifica se o texto contém conteúdo potencialmente perigoso."""
        for pattern in self.DANGEROUS_PATTERNS:
            if re.search(pattern, text, re.IGNORECASE):
                return True
        return False
    
    def add_disclaimer_if_needed(self, text: str, output_field: bool = False) -> str:
        """Adiciona disclaimer médico se for campo de resposta."""
        if output_field and len(text) > 100 and self.MEDICAL_DISCLAIMER not in text:
            self.stats['added_disclaimer'] += 1
            return text + self.MEDICAL_DISCLAIMER
        return text
    
    def sanitize_record(self, record: Dict) -> Optional[Dict]:
        """Sanitiza um registro completo."""
        self.stats['total_processed'] += 1
        sanitized = record.copy()
        
        # Verificar campos obrigatórios
        if not record.get('instruction') or not record.get('output'):
            self.stats['removed_empty'] += 1
            return None
        
        # Normalizar campos
        for field in ['instruction', 'input', 'output']:
            if field in sanitized and sanitized[field]:
                sanitized[field] = self.normalize_text(sanitized[field])
                self.stats['normalized'] += 1
        
        # Verificar comprimento mínimo da resposta
        output_words = len(sanitized['output'].split())
        if output_words < self.config['min_output_length']:
            self.stats['removed_empty'] += 1
            return None
        
        # Truncar resposta muito longa
        max_words = self.config['max_output_length'] // 5  # Aproximação
        sanitized['output'] = self.truncate_text(sanitized['output'], max_words)
        
        # Verificar conteúdo perigoso
        if self.check_dangerous_content(sanitized['output']):
            self.stats['flagged_dangerous'] += 1
            # Não removemos, apenas flagamos para revisão
            sanitized['_flagged'] = True
        
        return sanitized
    
    def get_report(self) -> Dict:
        """Retorna relatório de sanitização."""
        return self.stats.copy()

# Criar sanitizador
sanitizer = ContentSanitizer(CONFIG)
print("✅ Sanitizador de conteúdo configurado!")

### 5.2 Aplicar Sanitização

In [ ]:
print("🧹 Aplicando sanitização aos dados...\n")

sanitized_data = []
for record in tqdm(anonymized_data, desc="Sanitizando"):
    sanitized_record = sanitizer.sanitize_record(record)
    if sanitized_record is not None:
        sanitized_data.append(sanitized_record)

# Relatório de sanitização
sanit_report = sanitizer.get_report()

print("\n✅ Sanitização concluída!")
print(f"\n📊 Relatório de Sanitização:")
print(f"   - Registros processados: {sanit_report['total_processed']}")
print(f"   - Registros válidos: {len(sanitized_data)}")
print(f"   - Removidos (vazios): {sanit_report['removed_empty']}")
print(f"   - Truncados: {sanit_report['truncated']}")
print(f"   - Flagados para revisão: {sanit_report['flagged_dangerous']}")

---

## 6. Formatação para Hugging Face

### 6.1 Preparar Dataset

In [ ]:
def prepare_for_hf(records: List[Dict]) -> Dict[str, List]:
    """Prepara dados no formato esperado pelo Hugging Face datasets."""
    
    hf_data = {
        'instruction': [],
        'input': [],
        'output': [],
        'source': []
    }
    
    for record in records:
        hf_data['instruction'].append(record.get('instruction', ''))
        hf_data['input'].append(record.get('input', ''))
        hf_data['output'].append(record.get('output', ''))
        hf_data['source'].append(record.get('_source_file', 'unknown'))
    
    return hf_data

# Preparar dados para Hugging Face
print("📦 Preparando dados para formato Hugging Face...\n")

hf_data = prepare_for_hf(sanitized_data)

# Criar Dataset
dataset = Dataset.from_dict(hf_data)

print(f"✅ Dataset criado com sucesso!")
print(f"\n📊 Informações do Dataset:")
print(dataset)

### 6.2 Dividir em Train/Validation/Test

In [ ]:
# Calcular tamanhos dos splits
total_size = len(dataset)
train_size = int(total_size * CONFIG['train_split'])
val_size = int(total_size * CONFIG['val_split'])
test_size = total_size - train_size - val_size

print(f"📊 Divisão do Dataset:")
print(f"   - Total: {total_size}")
print(f"   - Train: {train_size} ({CONFIG['train_split']*100:.0f}%)")
print(f"   - Validation: {val_size} ({CONFIG['val_split']*100:.0f}%)")
print(f"   - Test: {test_size} ({CONFIG['test_split']*100:.0f}%)")

In [ ]:
# Realizar o split
dataset_shuffled = dataset.shuffle(seed=CONFIG['random_seed'])

# Split train + resto
train_test = dataset_shuffled.train_test_split(
    test_size=(1 - CONFIG['train_split']),
    seed=CONFIG['random_seed']
)

# Split validation + test
val_test_ratio = CONFIG['test_split'] / (CONFIG['val_split'] + CONFIG['test_split'])
val_test = train_test['test'].train_test_split(
    test_size=val_test_ratio,
    seed=CONFIG['random_seed']
)

# Criar DatasetDict
dataset_dict = DatasetDict({
    'train': train_test['train'],
    'validation': val_test['train'],
    'test': val_test['test']
})

print("\n✅ Dataset dividido com sucesso!")
print(f"\n📊 Tamanhos finais:")
for split_name, split_data in dataset_dict.items():
    print(f"   - {split_name}: {len(split_data)} registros")

---

## 7. Validação de Qualidade

### 7.1 Verificações de Integridade

In [ ]:
def validate_dataset(dataset: Dataset) -> Dict:
    """Valida a qualidade e integridade do dataset."""
    
    validation_results = {
        'total_records': len(dataset),
        'checks': {},
        'passed': True
    }
    
    # Check 1: Campos obrigatórios não vazios
    for field in ['instruction', 'output']:
        empty_count = sum(1 for item in dataset if not item[field] or len(item[field].strip()) == 0)
        passed = empty_count == 0
        validation_results['checks'][f'{field}_not_empty'] = {
            'passed': passed,
            'empty_count': empty_count
        }
        if not passed:
            validation_results['passed'] = False
    
    # Check 2: Comprimento mínimo de output
    short_outputs = sum(1 for item in dataset if len(item['output'].split()) < 10)
    passed = short_outputs == 0
    validation_results['checks']['min_output_length'] = {
        'passed': passed,
        'short_count': short_outputs
    }
    
    # Check 3: Sem PII residual (verificação básica)
    pii_patterns = [r'\d{3}\.\d{3}\.\d{3}-\d{2}', r'@.*\.com']  # CPF e email
    pii_found = 0
    for item in dataset:
        for pattern in pii_patterns:
            if re.search(pattern, item['output']):
                pii_found += 1
                break
    
    validation_results['checks']['no_pii_residual'] = {
        'passed': pii_found == 0,
        'pii_found': pii_found
    }
    
    # Check 4: Sem duplicatas
    instructions = [item['instruction'] for item in dataset]
    duplicates = len(instructions) - len(set(instructions))
    validation_results['checks']['no_duplicates'] = {
        'passed': duplicates == 0,
        'duplicate_count': duplicates
    }
    
    return validation_results

# Validar dataset completo
print("🔍 Validando qualidade do dataset...\n")

validation_results = validate_dataset(dataset)

print("📋 Resultados da Validação:")
print(f"   - Total de registros: {validation_results['total_records']}")
print(f"   - Status geral: {'✅ PASSOU' if validation_results['passed'] else '❌ FALHOU'}")
print("\n   Checks individuais:")

for check_name, result in validation_results['checks'].items():
    status = "✅" if result['passed'] else "❌"
    details = {k: v for k, v in result.items() if k != 'passed'}
    print(f"   {status} {check_name}: {details}")

### 7.2 Verificação de Amostras

In [ ]:
# Amostras aleatórias do dataset final
print("\n📝 Amostras Aleatórias do Dataset Final:\n")
print("=" * 80)

sample_indices = random.sample(range(len(dataset)), min(3, len(dataset)))

for idx in sample_indices:
    sample = dataset[idx]
    print(f"\n📄 Amostra #{idx}:")
    print("-" * 40)
    print(f"Fonte: {sample['source']}")
    print(f"Instruction: {sample['instruction'][:150]}..." if len(sample['instruction']) > 150 else f"Instruction: {sample['instruction']}")
    print(f"Input: {sample['input'] if sample['input'] else '(vazio)'}")
    print(f"Output: {sample['output'][:300]}..." if len(sample['output']) > 300 else f"Output: {sample['output']}")
    print("=" * 80)

---

## 8. Visualizações

### 8.1 Distribuição de Comprimentos

In [ ]:
# Calcular comprimentos
instruction_lengths = [len(item['instruction'].split()) for item in dataset]
output_lengths = [len(item['output'].split()) for item in dataset]

# Plotar distribuições
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de instruções
axes[0].hist(instruction_lengths, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(instruction_lengths), color='red', linestyle='--', label=f'Média: {np.mean(instruction_lengths):.1f}')
axes[0].set_xlabel('Número de Palavras')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de Comprimentos - Instructions')
axes[0].legend()

# Histograma de outputs
axes[1].hist(output_lengths, bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[1].axvline(np.mean(output_lengths), color='red', linestyle='--', label=f'Média: {np.mean(output_lengths):.1f}')
axes[1].set_xlabel('Número de Palavras')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição de Comprimentos - Outputs')
axes[1].legend()

plt.tight_layout()
plt.savefig(PROCESSED_DATA_DIR / 'length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Estatísticas de Comprimento:")
print(f"   Instructions: média={np.mean(instruction_lengths):.1f}, máx={max(instruction_lengths)}, mín={min(instruction_lengths)}")
print(f"   Outputs: média={np.mean(output_lengths):.1f}, máx={max(output_lengths)}, mín={min(output_lengths)}")

### 8.2 Distribuição por Fonte

In [ ]:
# Contagem por fonte
source_counts = Counter(item['source'] for item in dataset)

# Plotar
fig, ax = plt.subplots(figsize=(10, 6))

sources = list(source_counts.keys())
counts = list(source_counts.values())
colors = plt.cm.Set3(np.linspace(0, 1, len(sources)))

bars = ax.barh(sources, counts, color=colors, edgecolor='black')
ax.set_xlabel('Número de Registros')
ax.set_title('Distribuição de Registros por Fonte')

# Adicionar valores nas barras
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
            f'{count}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(PROCESSED_DATA_DIR / 'source_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.3 Word Cloud (se disponível)

In [ ]:
if WORDCLOUD_AVAILABLE:
    # Combinar todos os textos
    all_instructions = ' '.join(item['instruction'] for item in dataset)
    all_outputs = ' '.join(item['output'] for item in dataset)
    
    # Stopwords em português
    stopwords_pt = {'de', 'da', 'do', 'das', 'dos', 'a', 'o', 'e', 'é', 'em', 'para', 'com',
                    'que', 'um', 'uma', 'os', 'as', 'no', 'na', 'nos', 'nas', 'por', 'mais',
                    'se', 'ou', 'ao', 'aos', 'seu', 'sua', 'seus', 'suas', 'como', 'quando',
                    'qual', 'quais', 'este', 'esta', 'esse', 'essa', 'são', 'ser', 'pode',
                    'podem', 'deve', 'devem', 'foi', 'foram', 'está', 'estão', 'sobre', 'entre'}
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # WordCloud de instruções
    wc_instructions = WordCloud(width=800, height=400, background_color='white',
                                stopwords=stopwords_pt, colormap='Blues').generate(all_instructions)
    axes[0].imshow(wc_instructions, interpolation='bilinear')
    axes[0].axis('off')
    axes[0].set_title('Word Cloud - Instruções', fontsize=14)
    
    # WordCloud de outputs
    wc_outputs = WordCloud(width=800, height=400, background_color='white',
                           stopwords=stopwords_pt, colormap='Oranges').generate(all_outputs)
    axes[1].imshow(wc_outputs, interpolation='bilinear')
    axes[1].axis('off')
    axes[1].set_title('Word Cloud - Respostas', fontsize=14)
    
    plt.tight_layout()
    plt.savefig(PROCESSED_DATA_DIR / 'wordclouds.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ WordCloud não disponível. Instale com: pip install wordcloud")

### 8.4 Métricas de Qualidade

In [ ]:
# Calcular métricas de qualidade
quality_metrics = {
    'total_records': len(dataset),
    'sources': dict(source_counts),
    'instruction_stats': {
        'avg_words': float(np.mean(instruction_lengths)),
        'max_words': int(max(instruction_lengths)),
        'min_words': int(min(instruction_lengths)),
        'std_words': float(np.std(instruction_lengths))
    },
    'output_stats': {
        'avg_words': float(np.mean(output_lengths)),
        'max_words': int(max(output_lengths)),
        'min_words': int(min(output_lengths)),
        'std_words': float(np.std(output_lengths))
    },
    'anonymization': anon_report,
    'sanitization': sanit_report,
    'validation': validation_results,
    'splits': {
        'train': len(dataset_dict['train']),
        'validation': len(dataset_dict['validation']),
        'test': len(dataset_dict['test'])
    },
    'processing_timestamp': datetime.now().isoformat()
}

# Visualizar resumo
print("\n📊 RESUMO DE MÉTRICAS DE QUALIDADE")
print("=" * 50)
print(f"Total de registros: {quality_metrics['total_records']}")
print(f"\nDivisão dos dados:")
for split, count in quality_metrics['splits'].items():
    pct = count / quality_metrics['total_records'] * 100
    print(f"  - {split}: {count} ({pct:.1f}%)")
print(f"\nAnonimização:")
print(f"  - Entidades removidas: {quality_metrics['anonymization']['total_entities_anonymized']}")
print(f"\nSanitização:")
print(f"  - Registros processados: {quality_metrics['sanitization']['total_processed']}")
print(f"  - Registros removidos: {quality_metrics['sanitization']['removed_empty']}")

---

## 9. Exportação Final

### 9.1 Salvar Dataset Processado

In [ ]:
# Salvar dataset unificado em JSONL
print("💾 Salvando dataset processado...\n")

# Salvar dataset completo
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for item in dataset:
        record = {
            'instruction': item['instruction'],
            'input': item['input'],
            'output': item['output']
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"✅ Dataset salvo em: {OUTPUT_PATH}")
print(f"   - Tamanho: {OUTPUT_PATH.stat().st_size / 1024:.2f} KB")
print(f"   - Registros: {len(dataset)}")

In [ ]:
# Salvar splits separados (opcional)
for split_name, split_data in dataset_dict.items():
    split_path = PROCESSED_DATA_DIR / f"medical_data_{split_name}.jsonl"
    
    with open(split_path, 'w', encoding='utf-8') as f:
        for item in split_data:
            record = {
                'instruction': item['instruction'],
                'input': item['input'],
                'output': item['output']
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    
    print(f"✅ {split_name}: {split_path.name} ({len(split_data)} registros)")

### 9.2 Salvar Métricas e Relatórios

In [ ]:
# Salvar métricas
with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(quality_metrics, f, indent=2, ensure_ascii=False)

print(f"✅ Métricas salvas em: {METRICS_PATH}")

# Salvar relatório de sanitização
sanitization_full_report = {
    'timestamp': datetime.now().isoformat(),
    'anonymization': anon_report,
    'sanitization': sanit_report,
    'validation': validation_results
}

with open(SANITIZATION_REPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(sanitization_full_report, f, indent=2, ensure_ascii=False)

print(f"✅ Relatório de sanitização salvo em: {SANITIZATION_REPORT_PATH}")

### 9.3 Verificação Final

In [ ]:
# Verificar arquivos gerados
print("\n📁 Arquivos Gerados:")
print("=" * 50)

generated_files = list(PROCESSED_DATA_DIR.glob("*"))
for f in sorted(generated_files):
    size_kb = f.stat().st_size / 1024
    print(f"   ✅ {f.name} ({size_kb:.2f} KB)")

# Verificar integridade do arquivo principal
print(f"\n🔍 Verificando integridade de {OUTPUT_PATH.name}...")

try:
    with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
        loaded_count = sum(1 for line in f if line.strip())
    
    if loaded_count == len(dataset):
        print(f"   ✅ Arquivo válido: {loaded_count} registros")
    else:
        print(f"   ⚠️ Discrepância: esperado {len(dataset)}, encontrado {loaded_count}")
except Exception as e:
    print(f"   ❌ Erro ao verificar: {e}")

---

## 10. Conclusão e Próximos Passos

### 10.1 Resumo do Pipeline

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    📊 RESUMO DO PIPELINE DE PREPARAÇÃO                       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  1️⃣  CARREGAMENTO DE DADOS                                                   ║
║     • Arquivos JSONL carregados de múltiplas fontes                          ║
║     • Validação de formato e campos obrigatórios                             ║
║                                                                              ║
║  2️⃣  ANONIMIZAÇÃO (LGPD)                                                     ║
║     • Remoção de CPF, RG, CNS, telefones, emails                             ║
║     • Anonimização de nomes, datas, endereços                                ║
║     • Preservação do contexto médico relevante                               ║
║                                                                              ║
║  3️⃣  SANITIZAÇÃO                                                             ║
║     • Normalização de texto (espaços, encoding)                              ║
║     • Truncamento de respostas muito longas                                  ║
║     • Verificação de conteúdo potencialmente perigoso                        ║
║                                                                              ║
║  4️⃣  FORMATAÇÃO                                                              ║
║     • Conversão para formato Hugging Face Dataset                            ║
║     • Divisão train/validation/test                                          ║
║                                                                              ║
║  5️⃣  VALIDAÇÃO                                                               ║
║     • Verificação de campos não vazios                                       ║
║     • Verificação de PII residual                                            ║
║     • Detecção de duplicatas                                                 ║
║                                                                              ║
║  6️⃣  EXPORTAÇÃO                                                              ║
║     • Dataset unificado em JSONL                                             ║
║     • Splits separados (train/val/test)                                      ║
║     • Métricas e relatórios                                                  ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

print(f"\n📈 ESTATÍSTICAS FINAIS:")
print(f"   • Total de registros processados: {quality_metrics['total_records']}")
print(f"   • Entidades PII anonimizadas: {quality_metrics['anonymization']['total_entities_anonymized']}")
print(f"   • Split de treino: {quality_metrics['splits']['train']} registros")
print(f"   • Split de validação: {quality_metrics['splits']['validation']} registros")
print(f"   • Split de teste: {quality_metrics['splits']['test']} registros")
print(f"\n   • Arquivo principal: {OUTPUT_PATH}")

### 10.2 Próximos Passos

In [ ]:
print("""
🚀 PRÓXIMOS PASSOS:

1. FINE-TUNING DO MODELO
   • Carregar dataset com: dataset = load_dataset('json', data_files='medical_data_unified.jsonl')
   • Configurar modelo base (ex: Llama-2-7b-chat-hf)
   • Aplicar técnica LoRA/PEFT para fine-tuning eficiente
   • Treinar por 3 épocas com learning_rate=2e-4

2. AVALIAÇÃO
   • Usar split de teste para avaliação final
   • Calcular métricas: BLEU, ROUGE, perplexidade
   • Avaliação qualitativa por profissionais de saúde

3. INTEGRAÇÃO
   • Integrar com LangChain para orquestração
   • Implementar LangGraph para fluxos automatizados
   • Deploy do assistente virtual

4. MONITORAMENTO
   • Implementar logging de interações
   • Monitorar qualidade das respostas
   • Coletar feedback para melhoria contínua

📚 Documentação: README.md
📁 Código do pipeline: src/fine_tuning/data_preparation.py
""")

---

### ✅ Pipeline Concluído!

O dataset está pronto para ser utilizado no fine-tuning do modelo de linguagem. 
Todos os dados foram anonimizados conforme LGPD e validados quanto à qualidade.

**Desenvolvido para:** Tech Challenge Fase 3 - FIAP/Alura  
**Objetivo:** Assistente Virtual Médico Generalista  
**Conformidade:** LGPD (Lei Geral de Proteção de Dados)